# 예제1: Seq2Seq 기초 개념 및 간단한 영-한 번역
## 목표: Seq2Seq의 Encoder-Decoder 구조 이해

### 학습 내용
1. Seq2Seq 아키텍처 기초
2. Encoder 구현
3. Decoder 구현
4. 간단한 번역 실습

## STEP 1: 라이브러리 설치 및 데이터 준비

In [ ]:
# 필수 라이브러리
!pip install torch torchvision torchaudio -q
!pip install numpy pandas -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import random

# 재현성을 위한 시드 설정
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

## STEP 2: 간단한 데이터셋 생성

In [ ]:
# 간단한 영-한 번역 데이터
sentence_pairs = [
    ('hello', '안녕하세요'),
    ('good morning', '좋은 아침'),
    ('how are you', '당신은 어떻게 지내세요'),
    ('thank you', '감사합니다'),
    ('goodbye', '안녕히 가세요'),
    ('nice to meet you', '만나서 반갑습니다'),
    ('i love you', '나는 당신을 사랑합니다'),
    ('where is the bathroom', '화장실이 어디입니까'),
]

print(f"총 문장 쌍: {len(sentence_pairs)}개")
print("\n샘플 데이터:")
for eng, kor in sentence_pairs[:3]:
    print(f"  {eng} → {kor}")

## STEP 3: Vocabulary 생성

In [ ]:
# 영어 Vocabulary
eng_words = set()
for eng, _ in sentence_pairs:
    eng_words.update(eng.split())

# 한글 Vocabulary (character 기반)
kor_chars = set()
for _, kor in sentence_pairs:
    kor_chars.update(kor)

# 특수 토큰 추가
ENG_VOCAB = ['<PAD>', '<SOS>', '<EOS>', '<UNK>'] + sorted(list(eng_words))
KOR_VOCAB = ['<PAD>', '<SOS>', '<EOS>', '<UNK>'] + sorted(list(kor_chars))

# 단어-인덱스 매핑
eng_word2idx = {word: idx for idx, word in enumerate(ENG_VOCAB)}
eng_idx2word = {idx: word for word, idx in eng_word2idx.items()}

kor_char2idx = {char: idx for idx, char in enumerate(KOR_VOCAB)}
kor_idx2char = {idx: char for char, idx in kor_char2idx.items()}

print(f"영어 Vocabulary 크기: {len(ENG_VOCAB)}")
print(f"한글 Vocabulary 크기: {len(KOR_VOCAB)}")
print(f"\n영어 단어: {list(eng_words)[:5]}")
print(f"한글 문자: {list(kor_chars)[:10]}")

## STEP 4: Seq2Seq 모델 정의

In [ ]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embedding_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
    
    def forward(self, src):
        # src: (batch_size, seq_len)
        embedded = self.embedding(src)  # (batch_size, seq_len, embedding_dim)
        _, hidden = self.rnn(embedded)  # hidden: (1, batch_size, hidden_dim)
        return hidden

# Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, embedding_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, input, hidden):
        # input: (batch_size, 1)
        embedded = self.embedding(input)  # (batch_size, 1, embedding_dim)
        output, hidden = self.rnn(embedded, hidden)  # output: (batch_size, 1, hidden_dim)
        prediction = self.fc(output.squeeze(1))  # (batch_size, output_dim)
        return prediction, hidden

# Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # Encoder
        hidden = self.encoder(src)  # (1, batch_size, hidden_dim)
        
        # Decoder
        batch_size = tgt.size(0)
        max_len = tgt.size(1)
        
        # 첫 입력: <SOS> 토큰
        decoder_input = torch.tensor([1] * batch_size).unsqueeze(1).to(device)  # (batch_size, 1)
        
        outputs = []
        
        for t in range(max_len):
            prediction, hidden = self.decoder(decoder_input, hidden)
            outputs.append(prediction.unsqueeze(1))
            
            # Teacher forcing
            if random.random() < teacher_forcing_ratio and t < tgt.size(1) - 1:
                decoder_input = tgt[:, t].unsqueeze(1)
            else:
                decoder_input = prediction.argmax(1).unsqueeze(1)
        
        outputs = torch.cat(outputs, dim=1)  # (batch_size, max_len, output_dim)
        return outputs

print("✅ Seq2Seq 모델 정의 완료")

## STEP 5: 모델 초기화 및 학습

In [ ]:
# 하이퍼파라미터
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 50
BATCH_SIZE = 2
LEARNING_RATE = 0.001

# 모델 생성
encoder = Encoder(len(ENG_VOCAB), HIDDEN_DIM, EMBEDDING_DIM).to(device)
decoder = Decoder(len(KOR_VOCAB), HIDDEN_DIM, EMBEDDING_DIM).to(device)
model = Seq2Seq(encoder, decoder).to(device)

# 손실 함수 및 최적화기
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"✅ 모델 초기화 완료")
print(f"   Encoder 파라미터: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"   Decoder 파라미터: {sum(p.numel() for p in decoder.parameters()):,}")

## STEP 6: 데이터 전처리 함수

In [ ]:
def preprocess_sentence(sentence, word2idx, is_english=True):
    """문장을 인덱스로 변환"""
    if is_english:
        tokens = sentence.lower().split()
        indices = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    else:
        indices = [word2idx.get(char, word2idx['<UNK>']) for char in sentence]
    
    # <SOS>, <EOS> 추가
    indices = [word2idx['<SOS>']] + indices + [word2idx['<EOS>']]
    return indices

def pad_sequence(sequence, max_len, pad_idx):
    """시퀀스를 패딩"""
    if len(sequence) < max_len:
        sequence = sequence + [pad_idx] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]
    return sequence

# 최대 길이 계산
max_eng_len = max([len(eng.split()) for eng, _ in sentence_pairs]) + 2  # <SOS>, <EOS>
max_kor_len = max([len(kor) for _, kor in sentence_pairs]) + 2

print(f"영어 최대 길이: {max_eng_len}")
print(f"한글 최대 길이: {max_kor_len}")

## STEP 7: 학습

In [ ]:
# 데이터 전처리
src_data = []
tgt_data = []

for eng, kor in sentence_pairs:
    eng_indices = preprocess_sentence(eng, eng_word2idx, is_english=True)
    kor_indices = preprocess_sentence(kor, kor_char2idx, is_english=False)
    
    eng_padded = pad_sequence(eng_indices, max_eng_len, eng_word2idx['<PAD>'])
    kor_padded = pad_sequence(kor_indices, max_kor_len, kor_char2idx['<PAD>'])
    
    src_data.append(torch.tensor(eng_padded, dtype=torch.long))
    tgt_data.append(torch.tensor(kor_padded, dtype=torch.long))

src_data = torch.stack(src_data)
tgt_data = torch.stack(tgt_data)

print(f"소스 데이터 shape: {src_data.shape}")
print(f"타겟 데이터 shape: {tgt_data.shape}")

In [ ]:
# 학습
print("학습 시작...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for i in range(0, len(src_data), BATCH_SIZE):
        src_batch = src_data[i:i+BATCH_SIZE].to(device)
        tgt_batch = tgt_data[i:i+BATCH_SIZE].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(src_batch, tgt_batch)
        
        # 손실 계산
        loss = criterion(outputs.reshape(-1, len(KOR_VOCAB)), tgt_batch.reshape(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(src_data):.4f}")

print("✅ 학습 완료")

## STEP 8: 번역 테스트

In [ ]:
def translate(sentence, model, eng_word2idx, kor_idx2char, max_len):
    """문장 번역"""
    model.eval()
    
    # 입력 전처리
    src_indices = preprocess_sentence(sentence, eng_word2idx, is_english=True)
    src_padded = pad_sequence(src_indices, max_len, eng_word2idx['<PAD>'])
    src_tensor = torch.tensor(src_padded, dtype=torch.long).unsqueeze(0).to(device)
    
    # 번역
    with torch.no_grad():
        hidden = model.encoder(src_tensor)
        
        decoder_input = torch.tensor([1]).unsqueeze(0).to(device)  # <SOS>
        output_chars = []
        
        for _ in range(max_len):
            prediction, hidden = model.decoder(decoder_input, hidden)
            top_idx = prediction.argmax(1).item()
            
            if top_idx == 2:  # <EOS>
                break
            
            output_chars.append(kor_idx2char[top_idx])
            decoder_input = torch.tensor([top_idx]).unsqueeze(0).to(device)
    
    return ''.join(output_chars)

# 테스트
print("\n【번역 테스트】")
test_sentences = ['hello', 'thank you', 'goodbye']

for eng in test_sentences:
    translated = translate(eng, model, eng_word2idx, kor_idx2char, max_kor_len)
    print(f"영어: {eng}")
    print(f"번역: {translated}")
    print()

## 🎯 정리
- Seq2Seq의 Encoder-Decoder 구조 이해
- 간단한 번역 모델 구현 및 학습
- 번역 결과 확인